In [11]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import open3d as o3d
from scipy.spatial.transform import Rotation as R 
import os 
import glob 
import pandas as pd 

Thông số camera

In [ ]:
color_intrinsics = {
    'width': 1280,
    'height': 720,
    'fx': 643.90087890625,
    'fy': 643.1365356445312,
    'cx': 650.2113037109375,
    'cy': 355.79559326171875,
    'model': "distortion.inverse_brown_conrady",
    'coeffs': [-0.05658450722694397, 0.06544225662946701,-0.0008694113348610699, 0.00016751799557823688,-0.020957745611667633]
}

depth_intrinsics = {
    'width': 1280,
    'height': 720,
    'fx': 650.0616455078125,
    'fy': 650.0616455078125,
    'cx': 649.5928955078125,
    'cy': 360.9415588378906,
    'model': "distortion.brown_conrady",
    'coeffs': [0.0, 0.0, 0.0, 0.0, 0.0]
}

R_depth_to_color = np.array([

    [0.9999898076057434,    -0.00020347206736914814, -0.004507721401751041],
    [0.00018898719281423837, 0.9999948143959045,     -0.0032135415822267532],
    [0.004508351907134056,   0.003212657058611512,    0.9999846816062927]

    ])
t_depth_to_color = np.array([

    [-0.05905],
    [8.67399e-5],
    [0.00041]
    
    ])

In [24]:
def get_yolo_results_for_image(image_name, yolo_txt_dir):
    """
    Đọc file .txt kết quả YOLO tùy chỉnh và trích xuất bbox.
    Định dạng file dự kiến: {classname} {conf} {xmin} {xmax} {ymin} {ymax} ...
    
    Args:
        image_name (str): Tên file ảnh (ví dụ: "0000.png")
        yolo_txt_dir (str): Đường dẫn tới thư mục chứa file .txt
        
    Returns:
        list: Danh sách các bbox [[x_min, y_min, x_max, y_max], ...]
    """
    
    # 1. Tạo đường dẫn file .txt từ tên file ảnh
    base_name = os.path.splitext(image_name)[0]
    txt_name = base_name + ".txt"
    txt_path = os.path.join(yolo_txt_dir, txt_name)
    
    bboxes = []
    
    # 2. Kiểm tra file .txt có tồn tại không
    if not os.path.exists(txt_path):
        return []

    # 3. Đọc và xử lý file
    try:
        with open(txt_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                
                # 4. Parse 
                # {classname} {conf} {xmin} {xmax} {ymin} {ymax} ...
                parts = line.split()
                
                # Cần ít nhất 6 phần tử (class, conf, xmin, xmax, ymin, ymax)
                if len(parts) < 6:
                    print(f"  -> Cảnh báo: Dòng không hợp lệ trong {txt_name}: {line}")
                    continue
                    
                # 5. Trích xuất tọa độ pixel trực tiếp
                try:
                    x_min = int(float(parts[2]))
                    x_max = int(float(parts[3]))
                    y_min = int(float(parts[4]))
                    y_max = int(float(parts[5]))
                
                    # 6. Thêm vào list
                    bboxes.append([x_min, y_min, x_max, y_max])
                    
                except ValueError:
                    print(f"  -> Cảnh báo: Không thể parse tọa độ trong {txt_name}: {line}")

    except Exception as e:
        print(f"  -> Lỗi khi đọc file YOLO {txt_path}: {e}")
        return []
        
    return bboxes

In [ ]:
def get_point_cloud_from_bbox(bbox, depth_img, color_img, depth_intr, color_intr, R, t):
    """
    Trích xuất điểm 3D (trong hệ tọa độ color) từ một bounding box.
    
    Args:
        bbox: list [x_min, y_min, x_max, y_max]
        depth_img: Ảnh depth 
        color_img: Ảnh RGB 
        depth_intr, color_intr, R, t: Thông số camera
        
    Returns:
        (points_color_valid, colors_valid)
        points_color_valid: (N, 3) numpy array chứa các điểm 3D
        colors_valid: (N, 3) numpy array chứa màu (0-1)
    """
    box_x_min, box_y_min, box_x_max, box_y_max = map(int, bbox)
    
    # 1. Crop depth
    h_d_img, w_d_img = depth_img.shape
    box_x_min = max(0, box_x_min)
    box_y_min = max(0, box_y_min)
    box_x_max = min(w_d_img - 1, box_x_max)
    box_y_max = min(h_d_img - 1, box_y_max)

    if box_y_min >= box_y_max or box_x_min >= box_x_max:
        return np.array([]), np.array([]) # Box không hợp lệ

    bounding_box_z = depth_img[box_y_min:box_y_max+1, box_x_min:box_x_max+1]
    
    if bounding_box_z.size == 0:
         return np.array([]), np.array([])

    # 2. Back-project to 3D (Depth Frame)
    h, w = bounding_box_z.shape
    u, v = np.meshgrid(np.arange(w), np.arange(h))
    Z = bounding_box_z.astype(np.float32) / 1000.0  # meters
    
    valid_depth = (Z > 0.01) & (Z < 5.0) & np.isfinite(Z)
    
    u_full = u + box_x_min
    v_full = v + box_y_min
    
    X = (u_full - depth_intr['cx']) * Z / depth_intr['fx']
    Y = (v_full - depth_intr['cy']) * Z / depth_intr['fy']
    
    points_depth = np.stack((X, Y, Z), axis=-1).reshape(-1, 3)
    valid_flat = valid_depth.reshape(-1)
    points_depth_valid = points_depth[valid_flat]
    
    if len(points_depth_valid) == 0:
        return np.array([]), np.array([])

    # 3. Transform to Color Frame
    points_color = (R @ points_depth_valid.T).T + t.reshape(1, 3)
    
    # 4. Project to color image for color sampling
    Xc = points_color[:, 0]
    Yc = points_color[:, 1]
    Zc = points_color[:, 2]
    
    valid_z = Zc > 1e-6
    if not np.any(valid_z):
        return np.array([]), np.array([])

    Xc_v = Xc[valid_z]; Yc_v = Yc[valid_z]; Zc_v = Zc[valid_z]
    points_color_valid_z = points_color[valid_z]
    
    u_c = np.round(Xc_v * color_intr['fx'] / Zc_v + color_intr['cx']).astype(np.int32)
    v_c = np.round(Yc_v * color_intr['fy'] / Zc_v + color_intr['cy']).astype(np.int32)
    
    # 5. Sample colors
    h_c_img, w_c_img = color_img.shape[:2]
    in_bounds = (u_c >= 0) & (u_c < w_c_img) & (v_c >= 0) & (v_c < h_c_img)
    
    if not np.any(in_bounds):
        return np.array([]), np.array([])

    u_c_in = u_c[in_bounds]; v_c_in = v_c[in_bounds]
    points_color_valid = points_color_valid_z[in_bounds]
    colors_valid = color_img[v_c_in, u_c_in][..., ::-1] / 255.0 # Chuyển BGR (cv2) sang RGB
    
    return points_color_valid, colors_valid

In [26]:
# 1. TẠO DANH SÁCH ẢNH CẦN XỬ LÝ
base_path = r"C:\Users\ADMIN\OneDrive\Documents\ViettelAIRace_Lam\dataset\train"
rgb_files = sorted(glob.glob(os.path.join(base_path, "rgb", "*.png")))
depth_files = sorted(glob.glob(os.path.join(base_path, "depth", "*.png")))

YOLO_TXT_DIR = r"C:\Users\ADMIN\OneDrive\Documents\ViettelAIRace_Lam\source_haanh\yolo_result"

# 2. LIST ĐỂ LƯU KẾT QUẢ CUỐI CÙNG
all_final_outputs = []

# 3. VÒNG LẶP BÊN NGOÀI
for rgb_path, depth_path in zip(rgb_files, depth_files):
    
    IMAGE_FILENAME = os.path.basename(rgb_path)
    print(f"\n=================================================")
    print(f"ĐANG XỬ LÝ ẢNH: {IMAGE_FILENAME}")
    print(f"=================================================")

    depth = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
    img = cv2.imread(rgb_path, cv2.IMREAD_UNCHANGED)
    
    if depth is None or img is None:
        print(f"  -> Lỗi: Không load được ảnh {IMAGE_FILENAME}")
        continue
        
    all_yolo_bboxes = get_yolo_results_for_image(
        IMAGE_FILENAME, 
        YOLO_TXT_DIR
    )
    
    if not all_yolo_bboxes:
        print("  -> Không tìm thấy bounding box nào.")
        all_final_outputs.append((IMAGE_FILENAME, None, None, None))
        continue
        
    print(f"Bắt đầu Trích xuất 3D points cho {len(all_yolo_bboxes)} ứng cử viên...")

    candidate_parcels_info = []
    for i, bbox in enumerate(all_yolo_bboxes):
        try:
            points_3d, colors_3d = get_point_cloud_from_bbox(
                bbox, depth, img, 
                depth_intrinsics, color_intrinsics, 
                R_depth_to_color, t_depth_to_color
            )
            if len(points_3d) > 0:
                candidate_parcels_info.append({
                    'points_3d': points_3d,
                    'colors_3d': colors_3d,
                    'original_bbox': bbox
                })
        except Exception as e:
            print(f"  -> Lỗi khi xử lý bbox {bbox}: {e}")
            
    # --- BƯỚC 4: TÍNH TÂM 3D ---
    print("\nBắt đầu Tính toán Tâm 3D (lọc outlier + trung bình có trọng số)...")
    final_candidates = []

    for i, parcel_info in enumerate(candidate_parcels_info):
        parcel_points_3d = parcel_info['points_3d']
        if len(parcel_points_3d) < 100:
            print(f"  -> Bỏ qua ứng cử viên {i}: quá ít điểm.")
            continue

        # --- Lọc nhiễu không gian ---
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(parcel_points_3d)
        pcd_clean, ind = pcd.remove_statistical_outlier(nb_neighbors=30, std_ratio=1.0)
        filtered_points = np.asarray(pcd_clean.points)

        # --- Trung bình có trọng số ---
        mean = np.mean(filtered_points, axis=0)
        dist = np.linalg.norm(filtered_points - mean, axis=1)
        mask = dist < np.percentile(dist, 80)
        stable_points = filtered_points[mask]

        center_3d = np.mean(stable_points, axis=0)

        final_candidates.append({
            'center_3d': center_3d,
            'original_info': parcel_info
        })

        print(f"  Ứng cử viên {i}: Tâm 3D (sau lọc) = ({center_3d[0]:.3f}, {center_3d[1]:.3f}, {center_3d[2]:.3f})")

    # --- BƯỚC 5: CHỌN BƯU KIỆN CUỐI CÙNG ---
        print("\nBắt đầu Chọn bưu kiện cuối cùng...")
        top_parcel = None
        num_candidates = len(final_candidates)
        
        if num_candidates == 0:
            print("[KẾT QUẢ]: Không có ứng cử viên hợp lệ.")
            all_final_outputs.append((IMAGE_FILENAME, None, None, None)) # Chỉ 3 cột x,y,z
            continue # Thêm continue ở đây
            
        elif num_candidates == 1:
            top_parcel = final_candidates[0]
            print(f"[KẾT QUẢ]: Chọn ứng cử viên duy nhất.")
        else:
            print(f"Phát hiện {num_candidates} ứng cử viên. Áp dụng luật Tie-Break...")
            
            # ==================================================================
            # === SỬA LOGIC: "TRÊN CÙNG" = Z NHỎ NHẤT ===
            # ==================================================================
            
            # Sắp xếp theo Z TĂNG DẦN (reverse=False) để tìm vật GẦN NHẤT
            final_candidates.sort(key=lambda c: c['center_3d'][2], reverse=False) 
            
            # Đây là Z của vật gần nhất
            z_min = final_candidates[0]['center_3d'][2] 
            
            # Nhóm các bưu kiện có chênh lệch độ cao < 5mm
            # So sánh với z_min, không phải z_max
            tie_group = [c for c in final_candidates if abs(c['center_3d'][2] - z_min) < 0.005]
            print(f"  -> {len(tie_group)} bưu kiện có cùng độ cao GẦN NHẤT (±5mm).")
            # ==================================================================
            
            if len(tie_group) == 1:
                top_parcel = tie_group[0]
            else:
                # Ưu tiên bưu kiện xa robot nhất (|Y| lớn nhất)
                # Ràng buộc này trong đề bài là đúng, code bạn đã làm đúng
                tie_group.sort(key=lambda c: abs(c['center_3d'][1]), reverse=True)
                top_parcel = tie_group[0]
                print(f"  -> Chọn vật xa robot nhất (|Y| lớn nhất).")

        if top_parcel is not None:
            c = top_parcel['center_3d']
            print(f"\n--- [KẾT QUẢ CUỐI CÙNG CHO ẢNH {IMAGE_FILENAME}] ---")
            print(f"  Tâm gắp (x, y, z): ({c[0]:.4f}, {c[1]:.4f}, {c[2]:.4f})")
            # Chỉ lưu 3 cột x,y,z
            all_final_outputs.append((IMAGE_FILENAME, c[0], c[1], c[2]))
        else:
            print(f"\n[KẾT QUẢ]: Không chọn được bưu kiện nào.")
            # Chỉ lưu 3 cột x,y,z
            all_final_outputs.append((IMAGE_FILENAME, None, None, None))

print("\n=== HOÀN TẤT QUÁ TRÌNH ===")


ĐANG XỬ LÝ ẢNH: 0000.png
Bắt đầu Trích xuất 3D points cho 1 ứng cử viên...

Bắt đầu Tính toán Tâm 3D (lọc outlier + trung bình có trọng số)...
  Ứng cử viên 0: Tâm 3D (sau lọc) = (-0.119, 0.029, 1.072)

Bắt đầu Chọn bưu kiện cuối cùng...
[KẾT QUẢ]: Chọn ứng cử viên duy nhất.

--- [KẾT QUẢ CUỐI CÙNG CHO ẢNH 0000.png] ---
  Tâm gắp (x, y, z): (-0.1193, 0.0295, 1.0723)

ĐANG XỬ LÝ ẢNH: 0001.png
Bắt đầu Trích xuất 3D points cho 2 ứng cử viên...

Bắt đầu Tính toán Tâm 3D (lọc outlier + trung bình có trọng số)...
  Ứng cử viên 0: Tâm 3D (sau lọc) = (-0.119, 0.030, 1.073)

Bắt đầu Chọn bưu kiện cuối cùng...
[KẾT QUẢ]: Chọn ứng cử viên duy nhất.

--- [KẾT QUẢ CUỐI CÙNG CHO ẢNH 0001.png] ---
  Tâm gắp (x, y, z): (-0.1185, 0.0302, 1.0727)
  Ứng cử viên 1: Tâm 3D (sau lọc) = (-0.143, -0.169, 1.131)

Bắt đầu Chọn bưu kiện cuối cùng...
Phát hiện 2 ứng cử viên. Áp dụng luật Tie-Break...
  -> 1 bưu kiện có cùng độ cao GẦN NHẤT (±5mm).

--- [KẾT QUẢ CUỐI CÙNG CHO ẢNH 0001.png] ---
  Tâm gắp (x, y, z)

In [27]:
df = pd.DataFrame(all_final_outputs, 
                  columns=['image_filename', 'x', 'y', 'z'])

output_csv_path = "submission.csv"
df.to_csv(output_csv_path, index=False, float_format='%.6f')

print(f"Đã lưu kết quả ra file: {output_csv_path}")
df.head()


Đã lưu kết quả ra file: submission.csv


,image_filename,x,y,z
0,0000.png,-0.119349,0.029489,1.072290
1,0001.png,-0.118549,0.030183,1.072716
2,0001.png,-0.118549,0.030183,1.072716
3,0002.png,-0.163243,-0.150626,1.046128
4,0002.png,-0.163243,-0.150626,1.046128


In [28]:
gt = pd.read_csv(r"C:\Users\ADMIN\OneDrive\Documents\ViettelAIRace_Lam\dataset\train\Public_train.csv")
pred = pd.read_csv(r"C:\Users\ADMIN\OneDrive\Documents\ViettelAIRace_Lam\source_lam\submission.csv")

# chuẩn hoá tên trước khi merge:
pred['image_filename'] = pred['image_filename'].apply(
    lambda x: x if x.startswith('image_') else f"image_{x}"
)

merged = pd.merge(gt, pred, on='image_filename', suffixes=('_gt', '_pred'))

merged['err'] = np.sqrt(
    (merged['x_gt'] - merged['x_pred'])**2 +
    (merged['y_gt'] - merged['y_pred'])**2 +
    (merged['z_gt'] - merged['z_pred'])**2
)

merged['MCE_i'] = np.minimum(merged['err'] / 0.05, 1.0)

MCE = merged['MCE_i'].mean()

print(f"Mean Center Error (MCE): {MCE:.4f}")
print(f"→ Điểm tương ứng: {(1 - MCE)*100:.2f}%")

print("\nThống kê thêm:")
print(f"  Trung vị sai số: {merged['err'].median()*1000:.2f} mm")
print(f"  Trung bình sai số: {merged['err'].mean()*1000:.2f} mm")
print(f"  % ảnh hợp lệ (err ≤ 5cm): {(merged['err'] <= 0.05).mean()*100:.2f}%")


Mean Center Error (MCE): 0.9980
→ Điểm tương ứng: 0.20%

Thống kê thêm:
  Trung vị sai số: 73.06 mm
  Trung bình sai số: 105.53 mm
  % ảnh hợp lệ (err ≤ 5cm): 0.41%
